In [ ]:
import json
import os
import pandas as pd
import numpy as np

# Carga de datos.
print("Cargando dataset...")
csv_path = 'reglamento_nuevo_unificado.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")

career_code_mapping = {
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    'ELE-PLS13': 'Ing. Electromecánica', 'ELE-PLS23': 'Ing. Electromecánica',
    'INT9ELECTR': 'Ing. Electromecánica', 'INT9SDIGYT': 'Ing. Electromecánica',
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    'ECA-PLS13': 'Ing. Electrónica', 'ECA-PLS23': 'Ing. Electrónica', 'ECA9-OPT': 'Ing. Electrónica'
}

df_clean = df_raw.copy()
df_clean['Carrera'] = df_clean['Firma'].astype(str).str.strip().map(career_code_mapping)

s_nota = df_clean['Nota.Final'].fillna('')
df_clean['Rendio_1F'] = s_nota.str.contains('1F-').astype(int)
df_clean['Nota_1F'] = s_nota.str.extract(r'1F-(\d)')[0].astype(float)
df_clean['Aprobo_1F'] = (df_clean['Nota_1F'] >= 2).astype(int)

df_clean['Rendio_2F'] = s_nota.str.contains('2F-').astype(int)
df_clean['Nota_2F'] = s_nota.str.extract(r'2F-(\d)')[0].astype(float)
df_clean['Aprobo_2F'] = (df_clean['Nota_2F'] >= 2).astype(int)

df_clean['Nota_Num'] = (df_clean['Nota.Final'].astype(str).str.extractall(r'(\d+)')[0].groupby(level=0).last().astype(float))
df_clean.loc[df_clean['Nota.Final'].isna(), 'Nota_Num'] = np.nan
df_clean['Aprobo_Cualquiera'] = (df_clean['Nota_Num'] >= 2).astype(int)

# Formulación del Target_Convine_1F
# 1= El alumno tiene altas expectativas si va a rendir el 1F
# 0= El alumno se beneficia más si espera por el 2F o cree tener una esperanza baja en el 1F. 
df_clean['Target_Conviene_1F'] = np.where(
    df_clean['Aprobo_1F'] == 1, 1,
    np.where(
        (df_clean['Rendio_1F'] == 0) & (df_clean['Aprobo_2F'] == 1), 0,
        np.where(df_clean['Firma'] >= 60, 1, 0)
    )
)

print("Target distribution:")
print(df_clean['Target_Conviene_1F'].value_counts(normalize=True))
